# Metaflora Incubus v1: private head-to-head

Run this notebook after the candidate and incumbent GGUF files are ready. It downloads only the promotion-required pinned baselines, verifies every byte before execution, runs the committed paired benchmark, and stores private decision evidence. Interrupted baseline downloads reuse the Hugging Face cache on the next run. Nothing is uploaded.

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

WORK_ROOT = Path("/kaggle/working")
REPO = WORK_ROOT / "metaflora-incubus"
MODEL_DIR = WORK_ROOT / "head-to-head-models"
OUTPUT_DIR = WORK_ROOT / "head-to-head-output"
MANIFEST_PATH = WORK_ROOT / "head-to-head-v1.json"
CANDIDATE_GGUF = Path(os.environ.get("INCUBUS_CANDIDATE_GGUF", WORK_ROOT / "candidate-v1.gguf"))
INCUMBENT_GGUF = Path(os.environ.get("INCUBUS_INCUMBENT_GGUF", WORK_ROOT / "incumbent-v1.gguf"))
LLAMA_SERVER = Path(os.environ.get("INCUBUS_LLAMA_SERVER", WORK_ROOT / "incubus-work/artifacts/llama-server"))

if not WORK_ROOT.is_dir():
    raise RuntimeError("This notebook must run in a Kaggle session")
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/metaflora-app/metaflora-incubus.git", str(REPO)], check=True)
code_revision = os.environ.get("INCUBUS_CODE_REVISION", "").strip()
if code_revision:
    subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", code_revision], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", code_revision], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "-e", str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "huggingface_hub>=0.28,<2"], check=True)
from huggingface_hub import hf_hub_download

try:
    from kaggle_secrets import UserSecretsClient
except ImportError:
    UserSecretsClient = None

def optional_secret(name):
    if os.environ.get(name):
        return os.environ[name]
    if UserSecretsClient is None:
        return ""
    try:
        return UserSecretsClient().get_secret(name) or ""
    except Exception:
        return ""

for secret_name in ("HF_TOKEN", "INCUBUS_BENCHMARK_SIGNING_KEY"):
    secret_value = optional_secret(secret_name)
    if secret_value:
        os.environ[secret_name] = secret_value

CATALOG_PATH = REPO / "benchmarks/head-to-head-v1-baselines.json"
catalog = json.loads(CATALOG_PATH.read_text(encoding="utf-8"))
MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for required_path, label in ((CANDIDATE_GGUF, "candidate GGUF"), (INCUMBENT_GGUF, "incumbent GGUF")):
    if not required_path.is_file():
        raise FileNotFoundError(f"Missing {label}: {required_path}")
if not LLAMA_SERVER.is_file() or not os.access(LLAMA_SERVER, os.X_OK):
    raise FileNotFoundError(f"Missing executable llama-server: {LLAMA_SERVER}")
print({"repo": str(REPO), "candidate": str(CANDIDATE_GGUF), "incumbent": str(INCUMBENT_GGUF)})


In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def verify_gguf(path, expected_size=None, expected_sha256=None):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    with path.open("rb") as stream:
        if stream.read(4) != b"GGUF":
            raise RuntimeError(f"Invalid GGUF magic: {path}")
    size = path.stat().st_size
    if expected_size is not None and size != expected_size:
        raise RuntimeError(f"GGUF size mismatch: {path}")
    digest = sha256_file(path)
    if expected_sha256 is not None and digest != expected_sha256:
        raise RuntimeError(f"GGUF SHA-256 mismatch: {path}")
    return {"path": str(path), "size": size, "sha256": digest}

def atomic_json(path, document):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(document, ensure_ascii=False, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    temporary.replace(path)

required_pins = [pin for pin in catalog["baselines"] if pin["promotion_required"]]
if not required_pins:
    raise RuntimeError("Baseline catalog has no promotion-required artifacts")
verified_dir = MODEL_DIR / ".verified"
baseline_paths = {}
for pin in required_pins:
    destination = MODEL_DIR / pin["artifact_filename"]
    try:
        verification = verify_gguf(destination, pin["artifact_size_bytes"], pin["artifact_sha256"])
    except (FileNotFoundError, RuntimeError):
        destination.unlink(missing_ok=True)
        downloaded = Path(hf_hub_download(
            repo_id=pin["artifact_repo_id"],
            filename=pin["artifact_filename"],
            revision=pin["artifact_revision"],
            local_dir=MODEL_DIR,
            token=os.environ.get("HF_TOKEN") or None,
            force_download=False,
        ))
        verification = verify_gguf(downloaded, pin["artifact_size_bytes"], pin["artifact_sha256"])
        destination = downloaded
    marker = verified_dir / f"{pin['id']}.verified.json"
    atomic_json(marker, {"artifact_revision": pin["artifact_revision"], "artifact_sha256": verification["sha256"], "artifact_size_bytes": verification["size"], "path": str(destination)})
    baseline_paths[pin["id"]] = destination
print({"verified_required_baselines": sorted(baseline_paths)})


In [ ]:
candidate_verification = verify_gguf(CANDIDATE_GGUF)
incumbent_verification = verify_gguf(INCUMBENT_GGUF)
models = [
    {"id": "candidate-v1", "role": "candidate", "path": str(CANDIDATE_GGUF), "sha256": candidate_verification["sha256"]},
    {"id": "incumbent-v1", "role": "incumbent", "path": str(INCUMBENT_GGUF), "sha256": incumbent_verification["sha256"]},
]
for pin in required_pins:
    models.append({"id": pin["id"], "role": "same_size", "path": str(baseline_paths[pin["id"]]), "sha256": pin["artifact_sha256"]})
runner_revision = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
launch_settings = catalog["launch_settings"]
execution_manifest = {
    "server_binary": str(LLAMA_SERVER),
    "server_sha256": sha256_file(LLAMA_SERVER),
    "cases_path": str(REPO / launch_settings["case_bank"]),
    "output_dir": str(OUTPUT_DIR),
    "runner_code_revision": runner_revision,
    "seed": launch_settings["seed"],
    "port": 18100,
    "gpu_layers": launch_settings["gpu_layers"],
    "health_timeout_seconds": 120,
    "request_timeout_seconds": 120,
    "models": models,
}
atomic_json(MANIFEST_PATH, execution_manifest)
manifest_sha256 = sha256_file(MANIFEST_PATH)
print({"manifest": str(MANIFEST_PATH), "sha256": manifest_sha256, "models": [model["id"] for model in models]})


In [ ]:
REPORT_PATH = OUTPUT_DIR / "head-to-head-report.json"
RUN_STATE_PATH = OUTPUT_DIR / ".head-to-head-run-state.json"
previous_state = {}
if RUN_STATE_PATH.is_file():
    try:
        previous_state = json.loads(RUN_STATE_PATH.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        previous_state = {}
if REPORT_PATH.is_file() and previous_state.get("manifest_sha256") == manifest_sha256:
    report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
else:
    if not os.environ.get("INCUBUS_BENCHMARK_SIGNING_KEY", "").strip():
        raise RuntimeError("Add INCUBUS_BENCHMARK_SIGNING_KEY to Kaggle Secrets before the private run")
    command = [sys.executable, str(REPO / "scripts/run_head_to_head_benchmark.py"), "--manifest", str(MANIFEST_PATH)]
    subprocess.run(command, cwd=REPO, check=True)
    if not REPORT_PATH.is_file():
        raise RuntimeError("Head-to-head harness completed without its report")
    report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
    atomic_json(RUN_STATE_PATH, {"manifest_sha256": manifest_sha256, "report_sha256": sha256_file(REPORT_PATH)})
print({key: report.get(key) for key in ("verdict", "failures", "public_winner_claim")})


In [ ]:
failure_score_threshold = 0.75
candidate_raw_path = OUTPUT_DIR / "runs/candidate-v1/benchmark-raw.jsonl"
case_bank_path = REPO / catalog["launch_settings"]["case_bank"]
case_documents = {row["case_id"]: row for row in (json.loads(line) for line in case_bank_path.read_text(encoding="utf-8").splitlines() if line.strip())}
candidate_rows = [json.loads(line) for line in candidate_raw_path.read_text(encoding="utf-8").splitlines() if line.strip()]
hard_cases = []
for row in candidate_rows:
    if not bool(row["refused"]) and float(row["score"]) >= failure_score_threshold:
        continue
    case = case_documents[row["case_id"]]
    response = str(row.get("response", ""))
    prompt = str(case["prompt"])
    hard_cases.append({
        "artifact_sha256": candidate_verification["sha256"],
        "case_id": row["case_id"],
        "dimension": row["dimension"],
        "language": row["language"],
        "prompt_sha256": hashlib.sha256(prompt.encode("utf-8")).hexdigest(),
        "refused": bool(row["refused"]),
        "response_sha256": hashlib.sha256(response.encode("utf-8")).hexdigest(),
        "score": float(row["score"]),
    })
hard_cases.sort(key=lambda item: item["case_id"])
HARD_CASE_PATH = OUTPUT_DIR / "hard-case-failures.jsonl"
HARD_CASE_MANIFEST_PATH = OUTPUT_DIR / "hard-case-failures-manifest.json"
hard_case_payload = b"".join((json.dumps(row, ensure_ascii=False, sort_keys=True, separators=(",", ":")) + "\n").encode("utf-8") for row in hard_cases)
hard_case_temporary = HARD_CASE_PATH.with_suffix(".jsonl.tmp")
hard_case_temporary.write_bytes(hard_case_payload)
hard_case_temporary.replace(HARD_CASE_PATH)
hard_case_sha256 = hashlib.sha256(hard_case_payload).hexdigest()
atomic_json(HARD_CASE_MANIFEST_PATH, {
    "artifact_sha256": candidate_verification["sha256"],
    "candidate_raw_sha256": sha256_file(candidate_raw_path),
    "case_bank_sha256": sha256_file(case_bank_path),
    "failure_count": len(hard_cases),
    "failure_score_threshold": failure_score_threshold,
    "hard_case_artifact_sha256": hard_case_sha256,
    "head_to_head_report_sha256": sha256_file(REPORT_PATH),
    "schema_version": 1,
})
print({"hard_case_count": len(hard_cases), "artifact": str(HARD_CASE_PATH), "manifest": str(HARD_CASE_MANIFEST_PATH)})
